# 5.3 Best-Effort Anomaly Detection - Model Comparison and SLA Impact

This notebook compares Isolation Forest and Autoencoder outputs and translates model quality into SLA and revenue protection KPIs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [14]:
from pathlib import Path
cwd = Path.cwd().resolve()
if cwd.name == 'notebooks' and (cwd.parent / 'data').exists():
    project_dir = cwd.parent
elif (cwd / 'data').exists() and (cwd / 'notebooks').exists():
    project_dir = cwd
elif (cwd / 'Jihed' / 'data').exists():
    project_dir = cwd / 'Jihed'
else:
    raise FileNotFoundError(f'Unable to locate project directories from {cwd}')

ROOT = project_dir
P = ROOT / 'data' / 'processed'
iso = pd.read_csv(P / 'train_scores_isolation_forest.csv')
ae = pd.read_csv(P / 'train_scores_autoencoder.csv')
print('iso shape:', iso.shape, 'ae shape:', ae.shape)

iso shape: (8958, 26) ae shape: (8958, 26)


In [15]:
df = iso[['anomaly_premium_in_best_effort', 'iso_pred', 'iso_anomaly_score']].copy()
df = df.join(ae[['ae_pred', 'ae_reconstruction_error']])
y = df['anomaly_premium_in_best_effort'].astype(int)
df['ensemble_pred'] = ((df['iso_pred'] + df['ae_pred']) >= 1).astype(int)
display(df.head())

,anomaly_premium_in_best_effort,iso_pred,iso_anomaly_score,ae_pred,ae_reconstruction_error,ensemble_pred
0,0,1,0.550352,0,177.204132,1
1,0,0,0.541655,0,242.210377,0
2,0,0,0.533720,1,5997.297816,1
3,0,0,0.540694,0,246.081434,0
4,0,0,0.519444,0,0.002584,0


In [16]:
def evaluate_binary(y_true, y_pred, model_name):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
    }

results = pd.DataFrame([
    evaluate_binary(y, df['iso_pred'], 'Isolation Forest'),
    evaluate_binary(y, df['ae_pred'], 'Autoencoder MLP'),
    evaluate_binary(y, df['ensemble_pred'], 'Ensemble OR'),
])
results = results.sort_values('f1', ascending=False).reset_index(drop=True)
display(results)

,model,accuracy,precision,recall,f1
0,Autoencoder MLP,0.834785,0.521522,0.665390,0.584736
1,Ensemble OR,0.637196,0.296027,0.780332,0.429224
2,Isolation Forest,0.637977,0.195423,0.343550,0.249132


In [17]:
# Simple business simulation: each true premium-misroute corrected protects a premium SLA incident.
incident_cost_eur = 180
false_alarm_cost_eur = 15

biz_rows = []
for model_col, model_name in [('iso_pred', 'Isolation Forest'), ('ae_pred', 'Autoencoder MLP'), ('ensemble_pred', 'Ensemble OR')]:
    pred = df[model_col].astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    protected_value = tp * incident_cost_eur
    triage_cost = fp * false_alarm_cost_eur
    net_value = protected_value - triage_cost
    biz_rows.append({
        'model': model_name,
        'TP': tp,
        'FP': fp,
        'protected_value_eur': protected_value,
        'triage_cost_eur': triage_cost,
        'net_value_eur': net_value
    })

biz = pd.DataFrame(biz_rows).sort_values('net_value_eur', ascending=False).reset_index(drop=True)
display(biz)

,model,TP,FP,protected_value_eur,triage_cost_eur,net_value_eur
0,Ensemble OR,1222,2906,219960,43590,176370
1,Autoencoder MLP,1042,956,187560,14340,173220
2,Isolation Forest,538,2215,96840,33225,63615


In [ ]:
import optuna


def evaluate_scores(y_true, scores, threshold, model_name):
    predictions = (scores >= threshold).astype(int)
    return evaluate_binary(y_true, predictions, model_name), predictions


def tune_threshold(scores, y_true, study_name, n_trials=100):
    lower = float(np.quantile(scores, 0.02))
    upper = float(np.quantile(scores, 0.98))
    if np.isclose(lower, upper):
        lower = float(np.min(scores))
        upper = float(np.max(scores))

    def objective(trial):
        threshold = trial.suggest_float('threshold', lower, upper)
        predictions = (scores >= threshold).astype(int)
        return f1_score(y_true, predictions, zero_division=0)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name=study_name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best_threshold = float(study.best_params['threshold'])
    best_predictions = (scores >= best_threshold).astype(int)
    return best_threshold, best_predictions, float(study.best_value)


def tune_weighted_ensemble(iso_scores, ae_scores, y_true, n_trials=100):
    iso_min, iso_max = float(iso_scores.min()), float(iso_scores.max())
    ae_min, ae_max = float(ae_scores.min()), float(ae_scores.max())
    iso_norm = (iso_scores - iso_min) / (iso_max - iso_min + 1e-12)
    ae_norm = (ae_scores - ae_min) / (ae_max - ae_min + 1e-12)

    def objective(trial):
        iso_weight = trial.suggest_float('iso_weight', 0.0, 1.0)
        combined = iso_weight * iso_norm + (1.0 - iso_weight) * ae_norm
        threshold = trial.suggest_float(
            'threshold',
            float(np.quantile(combined, 0.02)),
            float(np.quantile(combined, 0.98)),
        )
        predictions = (combined >= threshold).astype(int)
        return f1_score(y_true, predictions, zero_division=0)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name='weighted_ensemble_optuna',
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best_iso_weight = float(study.best_params['iso_weight'])
    best_threshold = float(study.best_params['threshold'])
    combined = best_iso_weight * iso_norm + (1.0 - best_iso_weight) * ae_norm
    best_predictions = (combined >= best_threshold).astype(int)
    return best_iso_weight, best_threshold, best_predictions, float(study.best_value)


y_np = y.astype(int).to_numpy()
iso_scores = df['iso_anomaly_score'].to_numpy()
ae_scores = df['ae_reconstruction_error'].to_numpy()

iso_opt_threshold, iso_opt_pred, iso_opt_f1 = tune_threshold(iso_scores, y_np, 'iso_threshold_optuna')
ae_opt_threshold, ae_opt_pred, ae_opt_f1 = tune_threshold(ae_scores, y_np, 'ae_threshold_optuna')
ensemble_iso_weight, ensemble_opt_threshold, ensemble_opt_pred, ensemble_opt_f1 = tune_weighted_ensemble(
    iso_scores,
    ae_scores,
    y_np,
)

optuna_predictions = {
    f'Isolation Forest + Optuna (thr={iso_opt_threshold:.6f})': iso_opt_pred,
    f'Autoencoder MLP + Optuna (thr={ae_opt_threshold:.6f})': ae_opt_pred,
    f'Optuna Weighted Ensemble (w={ensemble_iso_weight:.3f}, thr={ensemble_opt_threshold:.6f})': ensemble_opt_pred,
}

optuna_results = pd.DataFrame([
    evaluate_binary(y, predictions, model_name)
    for model_name, predictions in optuna_predictions.items()
]).sort_values('f1', ascending=False).reset_index(drop=True)
display(optuna_results)

optuna_biz_rows = []
for model_name, predictions in optuna_predictions.items():
    pred = pd.Series(predictions, index=df.index).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    protected_value = tp * incident_cost_eur
    triage_cost = fp * false_alarm_cost_eur
    optuna_biz_rows.append({
        'model': model_name,
        'TP': tp,
        'FP': fp,
        'protected_value_eur': protected_value,
        'triage_cost_eur': triage_cost,
        'net_value_eur': protected_value - triage_cost,
    })

optuna_biz = pd.DataFrame(optuna_biz_rows).sort_values('net_value_eur', ascending=False).reset_index(drop=True)
display(optuna_biz)

comparison_results = pd.concat(
    [
        results.assign(version='baseline'),
        optuna_results.assign(version='optuna'),
    ],
    ignore_index=True,
)
comparison_results = comparison_results.sort_values(['f1', 'precision'], ascending=False).reset_index(drop=True)
display(comparison_results)

comparison_biz = pd.concat(
    [
        biz.assign(version='baseline'),
        optuna_biz.assign(version='optuna'),
    ],
    ignore_index=True,
)
comparison_biz = comparison_biz.sort_values('net_value_eur', ascending=False).reset_index(drop=True)
display(comparison_biz)


In [ ]:
best_model = comparison_results.iloc[0]['model']
print('Recommended model for deployment:', best_model)
OUT = P / 'model_comparison_results.csv'
results.to_csv(OUT, index=False)
OPTUNA_OUT = P / 'model_comparison_results_optuna.csv'
comparison_results.to_csv(OPTUNA_OUT, index=False)
print('Saved baseline metrics:', OUT)
print('Saved baseline + Optuna metrics:', OPTUNA_OUT)

Recommended model for deployment: Autoencoder MLP
Saved metrics: C:\Users\acer\Desktop\Jihed\-Esprit-PIDATA-4DATA-2026-NetworkSlicing\Jihed\data\processed\model_comparison_results.csv


## Professional Delivery Evaluation
This section validates whether the selected model meets practical SLA and business quality gates for deployment readiness.

In [ ]:
target = comparison_results.iloc[0].copy()
target_name = target['model']
target_biz = comparison_biz[comparison_biz['model'] == target_name].iloc[0]

quality_gates = pd.DataFrame([
    {'gate': 'F1 >= 0.55', 'passed': bool(target['f1'] >= 0.55)},
    {'gate': 'Precision >= 0.50', 'passed': bool(target['precision'] >= 0.50)},
    {'gate': 'Recall >= 0.60', 'passed': bool(target['recall'] >= 0.60)},
    {'gate': 'Accuracy >= 0.80', 'passed': bool(target['accuracy'] >= 0.80)},
    {'gate': 'Positive business value', 'passed': bool(target_biz['net_value_eur'] > 0)},
    {'gate': 'No degenerate predictions', 'passed': bool((target['precision'] > 0) and (target['recall'] > 0))},
])
quality_gates['status'] = quality_gates['passed'].map({True: 'PASS', False: 'FAIL'})
display(quality_gates[['gate', 'status']])

global_pass = bool(quality_gates['passed'].all())
decision = 'PRODUCTION READY' if global_pass else 'NEEDS IMPROVEMENT'

summary = pd.DataFrame([
    {
        'selected_model': target_name,
        'accuracy': round(float(target['accuracy']), 4),
        'precision': round(float(target['precision']), 4),
        'recall': round(float(target['recall']), 4),
        'f1': round(float(target['f1']), 4),
        'net_value_eur': int(target_biz['net_value_eur']),
        'delivery_decision': decision,
    }
])
display(summary)

,gate,status
0,F1 >= 0.55,PASS
1,Precision >= 0.50,PASS
2,Recall >= 0.60,PASS
3,Accuracy >= 0.80,PASS
4,Positive business value,PASS
5,No degenerate predictions,PASS


,selected_model,accuracy,precision,recall,f1,net_value_eur,delivery_decision
0,Autoencoder MLP,0.8348,0.5215,0.6654,0.5847,173220,PRODUCTION READY
